# Lab 25: The Juggling Limit

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 3, *The Attention Filter*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-25.ipynb)

**What you will do:** find your own multiple-object-tracking capacity by testing your accuracy at 2, 3, 4, 5 and 6 targets, compare where it collapses with the classic four-object limit, and watch a simple algorithmic tracker follow far more than four dots without breaking a sweat.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 25 you tried to track a subset of moving circles as they wove among identical
distractors (Figure 3.4). Tracking 2 or 3 targets was easy. At 4 it became hard, and at 5 or
more your performance collapsed -- you started losing targets and confusing them with
distractors. This roughly-four-object limit is remarkably consistent across people and tasks
(Pylyshyn & Storm, 1988), and it does not improve much with practice.

## Record your results

Run an online multiple-object-tracking demonstration at several target-set sizes -- for
example 2, 3, 4, 5 and 6 targets among a larger field of distractors. Do at least 5 trials at
each set size and record the proportion of targets you correctly identified at the end of
each trial.

In [ ]:
n_targets = [2, 3, 4, 5, 6]
my_accuracy = {2: 1.00, 3: 0.95, 4: 0.85, 5: 0.55, 6: 0.40}  # example data: replace with yours

results = pd.Series(my_accuracy, name="accuracy")
results.index.name = "number of targets"
display(results.to_frame())
results.plot(marker="o", figsize=(6, 3.5), color="steelblue")
plt.axvline(4, color="grey", linestyle="--", label="classic ~4-object limit")
plt.xlabel("number of targets tracked"); plt.ylabel("proportion correctly identified")
plt.ylim(0, 1.05); plt.legend(); plt.show()

## Compare

The book reports a tracking capacity of "roughly **4 objects**" (Pylyshyn & Storm, 1988) that
holds across people and does not expand with practice or motivation. Find where your own
accuracy starts to fall away from ceiling.

In [ ]:
near_ceiling = results[results >= 0.9].index
your_limit = near_ceiling.max() if len(near_ceiling) else min(n_targets)
print(f"You stayed at or above 90% accuracy up to {your_limit} targets (book's estimate: about 4).")
if abs(your_limit - 4) <= 1:
    print("Close to the classic ~4-object limit.")
else:
    print("Some distance from the classic ~4-object estimate. Set sizes, target speed and how close")
    print("together the dots move all shift the apparent limit (Scholl, 2009) -- note the demo's speed")
    print("and dot count alongside your numbers.")

## The AI side

AI multi-object trackers (SORT, DeepSORT, ByteTrack) solve the same problem your brain
solved in Step 2 -- keeping a consistent identity on each of several moving things -- by
combining position prediction with an optimal matching step. Below, a small self-contained
tracker follows far more than four simulated dots by matching each dot in one frame to its
nearest neighbour in the previous frame, using the Hungarian algorithm (the same optimal
assignment method real trackers use) instead of the brain's roughly four attentional
"indices".

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

rng = np.random.default_rng(2026)
n_dots, n_frames, canvas = 20, 60, 500.0

pos = rng.uniform(50, canvas - 50, size=(n_dots, 2))
vel = rng.uniform(-4, 4, size=(n_dots, 2))
trajectory = np.zeros((n_frames, n_dots, 2))
for t in range(n_frames):
    pos = pos + vel
    bounce = (pos < 0) | (pos > canvas)
    vel[bounce] *= -1
    trajectory[t] = pos

ids = np.zeros((n_frames, n_dots), dtype=int)
ids[0] = np.arange(n_dots)
for t in range(1, n_frames):
    prev, curr = trajectory[t - 1], trajectory[t]
    cost = np.linalg.norm(curr[:, None, :] - prev[None, :, :], axis=2)
    row, col = linear_sum_assignment(cost)   # optimal one-to-one matching, not a ~4-item limit
    assignment = np.empty(n_dots, dtype=int)
    assignment[row] = col
    ids[t] = ids[t - 1][assignment]

correct = (ids[-1] == np.arange(n_dots)).mean()
print(f"The tracker correctly re-identified {correct:.0%} of {n_dots} dots after {n_frames} frames.")
print("Your own tracking capacity, from Step 3 above, was roughly four objects at once.")

fig, ax = plt.subplots(figsize=(5, 5))
for d in range(n_dots):
    ax.plot(trajectory[:, d, 0], trajectory[:, d, 1], alpha=0.4)
    ax.scatter(*trajectory[-1, d], s=25)
ax.set_title(f"A simple algorithmic tracker following {n_dots} dots")
ax.set_xlim(0, canvas); ax.set_ylim(0, canvas); plt.show()

The tracker has no capacity limit like yours because it does not maintain a small number of
attentional "indices" (Pylyshyn, 1989) -- it recomputes the globally optimal match for every
dot, every frame, at a computational cost that grows with the number of dots. Real AI
trackers do have their own bottlenecks (occlusion, look-alike objects, frame rate), but the
bottleneck is computational cost and ambiguity, not a hard cap around four.

## Pool the class data

Five trials per set size gives a noisy curve for one person. Pool the class: each person adds
rows with an anonymous ID, the set size, and their accuracy, downloads the sheet as CSV, and
loads it here. The rows below are placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,n_targets,accuracy
P01,2,1.0
P01,3,0.9
P01,4,0.8
P01,5,0.5
P01,6,0.35
P02,2,1.0
P02,3,1.0
P02,4,0.9
P02,5,0.6
P02,6,0.45
P03,2,0.95
P03,3,0.85
P03,4,0.7
P03,5,0.45
P03,6,0.3
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
class_curve = df.groupby("n_targets").accuracy.agg(["mean", "sem"])
display(class_curve)
plt.errorbar(class_curve.index, class_curve["mean"], yerr=class_curve["sem"], marker="o", color="steelblue")
plt.axvline(4, color="grey", linestyle="--", label="classic ~4-object limit")
plt.xlabel("number of targets"); plt.ylabel("mean accuracy"); plt.ylim(0, 1.05); plt.legend(); plt.show()

## Questions to think about

1. At what set size did your own accuracy first drop noticeably below ceiling? How close was it to the book's "roughly four" estimate?
2. The book describes two competing accounts of the tracking mechanism -- a small number of passive "indices" that stick to objects (Pylyshyn, 1989) versus a small number of active attentional foci that follow each target (Cavanagh & Alvarez, 2005). Does anything in your own data (for example, how errors happened -- losing a target completely versus swapping two targets) favour one account over the other?
3. The AI Lab's tracker re-identifies dots using a globally optimal match computed fresh every frame. Why does the human visual system not do this, given that it would presumably raise your four-object limit?
4. Scholl (2009) notes the four-object limit "is not fixed in stone" -- it shrinks when targets move faster or closer together. If you reran Step 2 with faster-moving dots, what would you predict for your own capacity, and why?

## Challenge

Test whether the four-object limit depends on speed. Run the same MOT demonstration at a fixed
number of targets (for example 4) at a slow dot speed and again at a fast dot speed, and
compare your accuracy.

In [ ]:
# Example data: replace with your own accuracy at a fixed target count, run at two dot speeds.
my_speed_test = {"slow dots": 0.90, "fast dots": 0.55}  # example data, both at 4 targets

results = pd.Series(my_speed_test, name="accuracy at 4 targets")
display(results.to_frame())
print("If accuracy drops noticeably at the faster speed, that supports Scholl's (2009) point that")
print("the tracking limit is not a fixed count but interacts with how demanding the tracking is.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-25.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")